# Predicción de accidentalidad — notebook de partida

Este notebook arranca solo la parte de leaderboard (entrenar + generar `mi_prediccion.csv`).
El EDA, calidad de datos, ingeniería de características y el resto de las secciones del taller van en su propio notebook/informe — ver el PDF del taller y el README de este assignment.

In [ ]:
import sqlite3

import pandas as pd

DB_PATH = "data_accidentes.sqlite3"  # ruta local al archivo que descargaron
CUTOFF = "2019-10-01 00:00:00"

con = sqlite3.connect(DB_PATH)
clima = pd.read_sql("SELECT * FROM clima", con, parse_dates=["TW"])
accidentes = pd.read_sql("SELECT * FROM accidentes", con, parse_dates=["TW"])
raw = pd.read_sql("SELECT * FROM raw_accidentes", con, parse_dates=["TW"])
con.close()

print(clima["TW"].min(), clima["TW"].max(), len(clima))
print(accidentes["TW"].max())  # debe ser 2019-09-30 23:00:00 -- de ahí en adelante no hay etiquetas

## Construir el target

`clima` es el universo completo de (barrio, hora). El target es 1 si esa pareja está en `accidentes`, 0 si no.
Como `accidentes` solo tiene datos hasta el corte, todo lo que quede con `TW >= CUTOFF` es, por construcción, el conjunto que deben predecir (sin target).

In [ ]:
positivos = set(zip(accidentes["BARRIO"], accidentes["TW"]))
clima["target"] = [
    1 if (b, tw) in positivos else 0
    for b, tw in zip(clima["BARRIO"], clima["TW"])
]

train = clima[clima["TW"] < CUTOFF].copy()
test = clima[clima["TW"] >= CUTOFF].copy()  # target aquí siempre será 0 -- no los usen, es solo un placeholder

print("train:", train.shape, "positivos:", train["target"].mean())
print("test (a predecir):", test.shape)

## Features

TODO: variables temporales cíclicas, agregados históricos por barrio (calculados SOLO con datos anteriores a cada fila -- ver advertencia de fuga de información en el README), variables climáticas relevantes, etc.

In [ ]:
# TODO: ingeniería de características

## Entrenamiento y validación

TODO: partición temporal dentro de `train` (ver sección 4.6 del taller), balanceo, comparación de modelos.

In [ ]:
# TODO: entrenar el modelo final sobre todo `train`

## Generar la submission

`target` debe ser un score/probabilidad continuo, no una etiqueta dura.

In [ ]:
# test["target"] = modelo.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test["BARRIO"] + "|" + test["TW"].dt.strftime("%Y-%m-%d %H:%M:%S"),
    "target": test["target"],  # reemplazar por las probabilidades del modelo
})
submission.to_csv("mi_prediccion.csv", index=False)
submission.head()